In [94]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [95]:
# 1. 抓取目前工作目錄
def read_file(dic_name):
    current_dir = os.getcwd()
    # 2. 把 current_dir 放最前面，其餘維持你原本的相對路徑
    results_dir = os.path.join(
        current_dir,
        "..", 
        "data", 
        "results", 
        dic_name
    )
    #print("Results will be saved to:", results_dir)
    defense_adv = pd.read_csv(f"{results_dir}/defense_adv.csv")
    defense_clean = pd.read_csv(f"{results_dir}/defense_clean.csv")
    normal_adv = pd.read_csv(f"{results_dir}/normal_adv.csv")
    normal_clean = pd.read_csv(f"{results_dir}/normal_clean.csv")
    return defense_adv, defense_clean, normal_adv, normal_clean

## Single Model

In [99]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 四種資料的清單
defense_adv_list = []
defense_clean_list = []
normal_adv_list = []
normal_clean_list = []

# 分別儲存四種資料的 MAE / RMSE
defense_adv_maes, defense_adv_rmses     = [], []
defense_clean_maes, defense_clean_rmses = [], []
normal_adv_maes, normal_adv_rmses       = [], []
normal_clean_maes, normal_clean_rmses   = [], []

for i in range(1, 11):
    defense_adv, defense_clean, normal_adv, normal_clean = read_file(
        f"dataset_campus_processed_ep_0.2_attack_FGSM_defence_AT_{i}"
    )

    # append 到各自的 list
    defense_adv_list.append(defense_adv)
    defense_clean_list.append(defense_clean)
    normal_adv_list.append(normal_adv)
    normal_clean_list.append(normal_clean)

    # 依序計算並打印每一種的 MAE / RMSE
    for name, df, maes_list, rmses_list in [
        ("Defense Adv",   defense_adv,   defense_adv_maes,   defense_adv_rmses),
        ("Defense Clean", defense_clean, defense_clean_maes, defense_clean_rmses),
        ("Normal Adv",    normal_adv,    normal_adv_maes,    normal_adv_rmses),
        ("Normal Clean",  normal_clean,  normal_clean_maes,  normal_clean_rmses),
    ]:
        mae  = mean_absolute_error(df["y_true"], df["y_pred"])
        rmse = np.sqrt(mean_squared_error(df["y_true"], df["y_pred"]))
        maes_list.append(mae)
        rmses_list.append(rmse)
        #print(f"[Iteration {i}] {name} → MAE: {mae:.4f}, RMSE: {rmse:.4f}")

# 最後印出各種資料的平均指標
print("\n=== Average Metrics for Each ===")
for name, maes_list, rmses_list in [
    ("Defense Model | Adv",   defense_adv_maes,   defense_adv_rmses),
    ("Defense Model | Clean", defense_clean_maes, defense_clean_rmses),
    ("Normal  Model | Adv",    normal_adv_maes,    normal_adv_rmses),
    ("Normal  Model | Clean",  normal_clean_maes,  normal_clean_rmses),
]:
    avg_mae  = sum(maes_list) / len(maes_list)
    avg_rmse = sum(rmses_list) / len(rmses_list)
    print(f"{name:15s} → AVG MAE: {avg_mae:.4f}, AVG RMSE: {avg_rmse:.4f}")


=== Average Metrics for Each ===
Defense Model | Adv → AVG MAE: 14.3981, AVG RMSE: 20.9499
Defense Model | Clean → AVG MAE: 9.6060, AVG RMSE: 14.5207
Normal  Model | Adv → AVG MAE: 31.7570, AVG RMSE: 39.3651
Normal  Model | Clean → AVG MAE: 8.6045, AVG RMSE: 13.0262


## Simple Ensemble

In [84]:
import numpy as np
from numpy.linalg import lstsq
from sklearn.metrics import mean_absolute_error, mean_squared_error

class AdaptiveEnsemble:
    """
    自適應集成模型：
    根據過去 window_size 步的真實值和各基模型預測，動態更新權重，並生成當前預測。
    """
    def __init__(self, n_models, window_size=10, non_negative=True, normalize=True, regularization=0.0):
        self.n_models = n_models
        self.window_size = window_size
        self.non_negative = non_negative
        self.normalize = normalize
        self.lambda_reg = regularization
        # 初始均勻權重
        self.weights = np.ones(n_models) / n_models

    def update_weights(self, y_true_hist, preds_hist):
        X = np.array(preds_hist)  # shape=(window_size, n_models)
        y = np.array(y_true_hist)  # shape=(window_size,)
        XtX = X.T.dot(X)
        if self.lambda_reg > 0:
            XtX += self.lambda_reg * np.eye(self.n_models)
        Xty = X.T.dot(y)

        # 最小二乘求解
        w, *_ = lstsq(XtX, Xty, rcond=None)
        if self.non_negative:
            w = np.clip(w, 0, None)
        if self.normalize and w.sum() > 0:
            w = w / w.sum()
        self.weights = w
        return w

    def predict(self, preds_current):
        return np.dot(self.weights, preds_current)

    def adapt_and_predict(self, y_true_hist, preds_hist, preds_current):
        # 更新權重後返回當前集成預測
        self.update_weights(y_true_hist, preds_hist)
        return self.predict(preds_current)

if __name__ == "__main__":
    # 假設已有 read_file 函式，可讀取並返回多組 DataFrame
    defense_adv_list = []
    for i in range(10):
        defense_adv, defense_clean, normal_adv, normal_clean = \
            read_file(f"dataset_campus_processed_ep_0.1_attack_FGSM_defence_AT_{i+1}")
        defense_adv_list.append(defense_adv)
    for i in range(10):
        defense_adv, defense_clean, normal_adv, normal_clean = \
            read_file(f"dataset_campus_processed_ep_0.05_attack_FGSM_defence_AT_{i+1}")
        defense_adv_list.append(defense_adv)


    # 提取真實值與預測矩陣
    y_true = defense_adv_list[0]['y_true'].values
    preds_matrix = np.vstack([df['y_pred'].values for df in defense_adv_list]).T
    n_samples = len(y_true)
    window_size = 2

    # 初始化集成器
    ensemble = AdaptiveEnsemble(
        n_models=len(defense_adv_list),
        window_size=window_size,
        non_negative=True,
        normalize=True,
        regularization=0.1
    )

    # 存儲集成預測與權重歷史
    ensemble_preds = np.zeros(n_samples)
    weights_history = []

    # 前 window_size 步先用均值並記錄初始權重
    ensemble_preds[:window_size] = preds_matrix[:window_size].mean(axis=1)
    weights_history.extend([ensemble.weights.copy() for _ in range(window_size)])

    # 滑動窗口自適應更新
    for t in range(window_size, n_samples):
        y_hist = y_true[t-window_size:t]
        preds_hist = preds_matrix[t-window_size:t, :]
        curr_preds = preds_matrix[t, :]
        # 更新並預測
        ensemble_preds[t] = ensemble.adapt_and_predict(y_hist, preds_hist, curr_preds)
        # 記錄當前更新後的權重
        weights_history.append(ensemble.weights.copy())

    # 評估
    mae = mean_absolute_error(y_true[window_size:], ensemble_preds[window_size:])
    rmse = np.sqrt(mean_squared_error(y_true[window_size:], ensemble_preds[window_size:]))
    print(f"Ensemble MAE: {mae}")
    print(f"Ensemble RMSE: {rmse}")

    # 輸出權重歷史：weights_history[t] 對應於時間步 t 的權重向量
    # 例如，最後一步的權重：
    print("最後一步權重：", weights_history[-1])
    # 或存成 numpy array 方便後續分析
    weights_history = np.array(weights_history)
    print("權重歷史形狀：", weights_history.shape)


Ensemble MAE: 10.098908392933087
Ensemble RMSE: 14.87786373895292
最後一步權重： [0.06886599 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.0182569  0.18544127
 0.26770244 0.         0.         0.3198389  0.02120128 0.04035666
 0.         0.07833657]
權重歷史形狀： (644, 20)
